# validate_stage5
CPU coordinator. Run cells 2, 3 and 4 in order. Supply the synthesis reference and evidence-linked review. Run after reviewing the synthesis.


In [ ]:
import json, os, sys, subprocess
from pathlib import Path
if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError('Use Python 3.11/3.12 with the pinned CPU requirements.')
for name in ('OPENBLAS_NUM_THREADS', 'OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[name] = '1'
from google.colab import drive
drive.mount('/content/drive')
PHASE1_ROOT = Path('/content/drive/MyDrive/phase1')
REPO = Path('/content/Research-stage5')
CODE_REF = 'phase1'  # Resolved once below; set this to the approved Stage 5 commit to pin explicitly.
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/Drazr/Research.git',str(REPO)],check=True)
if subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip():
    raise RuntimeError('Use a clean checkout; preserve local work before continuing.')
subprocess.run(['git','fetch','origin',CODE_REF],cwd=REPO,check=True)
CODE_COMMIT = subprocess.check_output(['git','rev-parse','FETCH_HEAD'],cwd=REPO,text=True).strip()
subprocess.run(['git','checkout','--detach',CODE_COMMIT],cwd=REPO,check=True)
subprocess.run([sys.executable,'-m','pip','install','-r',str(REPO/'requirements-stage2-cpu.txt')],check=True)
print('Fixed code commit:',CODE_COMMIT)
print('Restart if installation requests it, then rerun setup before continuing.')


In [ ]:
STAGE4_RELEASE = {'path': 'REPLACE_STAGE4_RELEASE/_COMPLETE.json', 'sha256': 'REPLACE_SHA256'}
SYNTHESIS_REF = {'path': 'REPLACE_SYNTHESIS/_COMPLETE.json', 'sha256': 'REPLACE_SHA256'}
REVIEW = PHASE1_ROOT / 'REPLACE_REVIEW.json'


In [ ]:
if subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() != CODE_COMMIT:
    raise RuntimeError('Checkout changed after setup.')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
    raise RuntimeError('Use a clean Stage 4 checkout.')
command = [sys.executable, '-u', '-m', 'src.stage5.operations',
           '--phase1-root', str(PHASE1_ROOT), '--release', json.dumps(STAGE4_RELEASE),
           '--storage-scope', 'persistent-drive', 'finalize']
subprocess.run(command + ['--synthesis', json.dumps(SYNTHESIS_REF), '--review', str(REVIEW)], cwd=REPO, env=os.environ.copy(), check=True)
